# 02 · Score: rules baseline + gradient boosting, validated vs the legacy engine
Input `gold_features` (from 01). Attach **lh_fraud** as default lakehouse, then **Run all**.

Outputs: `gold_scores`, `gold_alert_queue`, `gold_alert_explanations`, `gold_model_metrics`,
`gold_model_recall_by_type`, `Files/submission/submission.csv`.

- Time split from notebook 01: train = Jan 2025 → May 2026, valid = Jun–Jul 2026, test = Aug–Sep 2026 (no labels).
- Memory: train keeps **all frauds + 10 % of legit rows** (weight 10 restores the real imbalance); valid and test are kept **complete**, so validation numbers are honest.
- Alert budget = analyst capacity: top 1 % per period. Model scores, rules explain, the LLM (later) only talks.

In [ ]:
import json
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
LEGIT_FRAC = 0.10
SEED = 42

g = spark.table("gold_features")
tr = g.filter(F.col("period") == "train")
tr = (tr.filter(F.col("is_fraud") == 1).withColumn("sample_w", F.lit(1.0))
        .unionByName(tr.filter(F.col("is_fraud") == 0).sample(fraction=LEGIT_FRAC, seed=SEED)
                       .withColumn("sample_w", F.lit(1.0 / LEGIT_FRAC))))
rest = g.filter(F.col("period") != "train").withColumn("sample_w", F.lit(1.0))
df = tr.unionByName(rest).toPandas()

fcols = [c for c in df.columns if c.startswith("f_")]
df[fcols] = df[fcols].astype("float32").replace([np.inf, -np.inf], np.nan)   # ratios / z-scores can divide by 0
df["is_fraud"] = df["is_fraud"].fillna(0).astype(int)
df = df.sort_values(["period", "timestamp"]).reset_index(drop=True)
TR, VA, TE = (df.period == "train").values, (df.period == "valid").values, (df.period == "test").values
print(df.period.value_counts().to_dict(), "| frauds train:", int(df.loc[TR, "is_fraud"].sum()),
      "| frauds valid:", int(df.loc[VA, "is_fraud"].sum()), "|", len(fcols), "features")

## 1. Rules score (plan B + explanation source)
Each rule = one sentence an analyst understands. Negative points = reassuring signals (answer "why is this NOT fraud?").

In [ ]:
d = df
RULES = [  # (factor, condition, points, observed column, usual column)
    ("Montant très supérieur aux habitudes", (d.f_amount_ratio >= 3) & (d.f_n_before >= 5), 2, "f_amount", "f_avg_amt_before"),
    ("Premier paiement physique dans ce pays", (d.f_new_country_phys == 1) & (d.f_is_foreign == 1) & (d.f_n_before >= 5), 3, "f_amount", None),
    ("Déplacement impossible depuis l'opération précédente", d.f_speed_kmh > 900, 4, "f_speed_kmh", "f_dist_prev_km"),
    ("Rafale de transactions sur la carte (1 h)", d.f_key_tx_1h >= 4, 2, "f_key_tx_1h", None),
    ("Refus récents sur la carte (1 h)", d.f_key_declines_1h >= 2, 2, "f_key_declines_1h", None),
    ("Micro-montants répétés (test de carte)", d.f_key_small_tx_1h >= 3, 2, "f_key_small_tx_1h", None),
    ("Piste magnétique utilisée", d.f_magstripe == 1, 1, None, None),
    ("Opération de nuit inhabituelle pour ce client", d.f_unusual_night == 1, 1, "f_hour", "f_night_share_before"),
    ("Bénéficiaire ajouté moins d'1 h avant le virement", d.f_mins_since_benef_added <= 60, 3, "f_mins_since_benef_added", None),
    ("Échecs de connexion dans les 24 h", d.f_login_fail_24h >= 2, 2, "f_login_fail_24h", None),
    ("Nouvel appareil enrôlé dans les 24 h", (d.f_device_added_24h >= 1) & (d.f_new_device == 1), 1, "f_device_added_24h", None),
    ("Numéro de téléphone modifié dans les 24 h", d.f_phone_change_24h >= 1, 2, "f_phone_change_24h", None),
    ("Sans contact juste sous le plafond chez un nouveau commerçant", (d.f_contactless_near_limit == 1) & (d.f_new_merchant == 1), 2, "f_amount", None),
    ("Commerçant lié à des fraudes signalées (30 j)", d.f_merchant_frauds_30d >= 2, 2, "f_merchant_frauds_30d", None),
    ("Appareil lié à des fraudes signalées (30 j)", d.f_device_frauds_30d >= 1, 3, "f_device_frauds_30d", None),
    ("Carte nationale utilisée à l'étranger", d.f_card_not_intl_abroad == 1, 3, None, None),
    # reassuring
    ("Rassurant : carte à puce à l'étranger", d.f_chip_abroad == 1, -1, None, None),
    ("Rassurant : séjour cohérent (≥ 3 opérations dans ce pays sur 7 j)", (d.f_is_foreign == 1) & (d.f_same_country_7d >= 3), -2, "f_same_country_7d", None),
    ("Rassurant : événements sensibles faits en agence", d.f_branch_events_24h >= 1, -1, "f_branch_events_24h", None),
]
hits = np.column_stack([r[1].fillna(False).values.astype(bool) for r in RULES])
points = np.array([r[2] for r in RULES])
df["score_rules"] = (hits * points).sum(axis=1)
print("rules PR-AUC on valid:", round(average_precision_score(df.loc[VA, "is_fraud"], df.loc[VA, "score_rules"]), 4))

## 2. Gradient boosting (time split; imbalance handled with sample weights)
`gender`, ids, `fraud_scenario`, `fraud_reported_date` and `is_fraud` are never features.

In [ ]:
CATS = ["channel", "entry_mode", "transaction_type", "merchant_category", "segment", "kyc_risk_level",
        "card_type", "merchant_type", "decline_reason"]
NUM = fcols
FEATS = NUM + CATS
X = df[FEATS].copy()
for c in CATS:
    X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan).astype("float32")
cat_mask = [c in CATS for c in FEATS]
assert not any(k in c for c in FEATS for k in ["is_fraud", "scenario", "reported", "gender"]), "leak column in features"

def fit(mask):
    y = df.loc[mask, "is_fraud"].values
    sw = df.loc[mask, "sample_w"].values
    ratio = sw[y == 0].sum() / max(y.sum(), 1)          # real imbalance (sampling undone)
    w = sw * np.where(y == 1, min(ratio, 50.0), 1.0)
    m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
                                       l2_regularization=1.0, categorical_features=cat_mask,
                                       early_stopping=False, random_state=SEED)
    return m.fit(X[mask], y, sample_weight=w)

model_tr = fit(TR)
df["score_model"] = model_tr.predict_proba(X)[:, 1]   # train rows = in-sample, valid rows = honest out-of-time

## 3. Validation Jun–Jul 2026: model vs rules vs legacy engine (headline numbers for the jury)

In [ ]:
legacy = spark.table("legacy_rule_alerts_train").select("transaction_id").distinct().toPandas()
df["legacy_alert"] = df.transaction_id.isin(set(legacy.transaction_id)).astype(int)

v = df[VA]
y = v.is_fraud.values

def at_k(score, k):
    idx = np.argsort(-score, kind="stable")[:k]
    tp = int(y[idx].sum())
    return round(tp / max(k, 1), 3), round(tp / max(y.sum(), 1), 3)

k_leg = int(v.legacy_alert.sum())
M = {"valid_rows": len(v), "valid_frauds": int(y.sum()),
     "pr_auc_model": round(average_precision_score(y, v.score_model), 4),
     "pr_auc_rules": round(average_precision_score(y, v.score_rules), 4),
     "legacy_alerts": k_leg,
     "legacy_precision": round(float(v.loc[v.legacy_alert == 1, "is_fraud"].mean()), 3) if k_leg else 0.0,
     "legacy_recall": round(float(v.loc[v.is_fraud == 1, "legacy_alert"].mean()), 3)}
for pct in [0.005, 0.01, 0.02]:
    k = max(int(len(v) * pct), 1)
    tag = f"top{str(pct * 100).rstrip('0').rstrip('.')}pct"
    M[f"model_{tag}_precision"], M[f"model_{tag}_recall"] = at_k(v.score_model.values, k)
    M[f"rules_{tag}_precision"], M[f"rules_{tag}_recall"] = at_k(v.score_rules.values, k)
if k_leg:
    M["model_same_budget_as_legacy_precision"], M["model_same_budget_as_legacy_recall"] = at_k(v.score_model.values, k_leg)
    M["false_alerts_avoided_vs_legacy"] = int(k_leg * (M["model_same_budget_as_legacy_precision"] - M["legacy_precision"]))
print(json.dumps(M, indent=1))

k1 = max(int(len(v) * 0.01), 1)
top = v.nlargest(k1, "score_model")
per_type = (v[v.is_fraud == 1].groupby("fraud_scenario").transaction_id.count().to_frame("frauds")
             .join(top[top.is_fraud == 1].groupby("fraud_scenario").transaction_id.count().to_frame("caught_top1pct"))
             .join(v[(v.is_fraud == 1) & (v.legacy_alert == 1)].groupby("fraud_scenario").transaction_id.count()
                    .to_frame("caught_legacy"))
             .fillna(0).astype(int))
per_type["recall_model"] = (per_type.caught_top1pct / per_type.frauds).round(3)
per_type["recall_legacy"] = (per_type.caught_legacy / per_type.frauds).round(3)
print(per_type.sort_values("frauds", ascending=False).to_string())

## 4. Refit on train + valid, score Aug–Sep, choose alerts (analyst capacity = 1 %)

In [ ]:
model_full = fit(TR | VA)
df.loc[TE, "score_model"] = model_full.predict_proba(X[TE])[:, 1]

df["score_source"] = np.select([TR, VA, TE], ["in_sample", "validation", "test"], "other")
df["rank_in_period"] = df.groupby("period").score_model.rank(ascending=False, method="first").astype(int)
budget = df.groupby("period").transaction_id.transform(lambda s: max(int(len(s) * 0.01), 1))
df["is_alert"] = ((df.rank_in_period <= budget) & ~TR).astype(int)          # no alerts on the (sampled) train rows
df["risk_level"] = np.select([(df.is_alert == 1) & (df.rank_in_period <= budget * 0.3), df.is_alert == 1],
                             ["ELEVE", "MOYEN"], "FAIBLE")
print("alerts:", df.groupby("period").is_alert.sum().to_dict())

## 5. Explanations: 3–5 factors per alert, observed vs usual (what the copilot reads) + 3 demo cases

In [ ]:
va = df[VA]
obvious = va[va.is_fraud == 1].nlargest(1, "score_model").transaction_id.tolist()
subtle = (va[(va.is_fraud == 1) & (va.is_alert == 1) & (va.score_rules <= 1)]
          .nlargest(1, "score_model").transaction_id.tolist())
avoided_fp = va[(va.legacy_alert == 1) & (va.is_fraud == 0)].nsmallest(1, "score_model").transaction_id.tolist()
DEMO = {"obvious_fraud": obvious, "subtle_fraud": subtle, "avoided_false_positive": avoided_fp}
DEMO_IDS = set(obvious + subtle + avoided_fp)
print("DEMO:", DEMO)

expl_mask = ((df.is_alert == 1) | df.transaction_id.isin(DEMO_IDS)).values
rows = []
for i in np.where(expl_mask)[0]:
    r = df.iloc[i]
    fired = sorted(np.where(hits[i])[0], key=lambda j: -abs(points[j]))
    for j in fired[:5]:
        name, _, pts, obs, usual = RULES[j]
        rows.append({"transaction_id": r.transaction_id, "factor": name, "points": float(pts),
                     "observed": None if obs is None or pd.isna(r[obs]) else round(float(r[obs]), 3),
                     "usual": None if usual is None or pd.isna(r[usual]) else round(float(r[usual]), 3),
                     "source": "rules"})

# Model-side explanation (Gold level): SHAP top-3 drivers per explained transaction. Skipped if unavailable.
try:
    import shap
    sub = X[expl_mask]
    sv = shap.TreeExplainer(model_full).shap_values(sub)
    sv = sv[1] if isinstance(sv, list) else sv
    ids = df.loc[expl_mask, "transaction_id"].values
    for i in range(len(sub)):
        for j in np.argsort(-np.abs(sv[i]))[:3]:
            val = sub.iloc[i, j]
            rows.append({"transaction_id": ids[i], "factor": FEATS[j], "points": round(float(sv[i][j]), 4),
                         "observed": None if pd.isna(val) else round(float(val), 3), "usual": None,
                         "source": "shap"})
    print("SHAP added")
except Exception as e:
    print("SHAP skipped:", type(e).__name__, str(e)[:120])

expl = pd.DataFrame(rows, columns=["transaction_id", "factor", "points", "observed", "usual", "source"])
for c in ["points", "observed", "usual"]:
    expl[c] = pd.to_numeric(expl[c], errors="coerce").astype(float)
print(len(expl), "explanation rows for", int(expl_mask.sum()), "transactions")

## 6. Write Gold tables + submission file
Test rows never carry labels (nulled in Spark before writing).

In [ ]:
def save(pdf, name, null_labels=False):
    sdf = spark.createDataFrame(pdf)
    if null_labels:
        is_test = F.col("period") == "test"
        sdf = (sdf.withColumn("is_fraud", F.when(~is_test, F.col("is_fraud")).cast("int"))
                  .withColumn("fraud_scenario", F.when(~is_test, F.col("fraud_scenario"))))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)

keep = ~TR   # valid + test fully scored; train rows are a sample and in-sample, not useful downstream
save(df.loc[keep, ["transaction_id", "period", "score_source", "score_model", "score_rules", "rank_in_period",
                   "is_alert", "risk_level", "legacy_alert"]], "gold_scores")

top_factors = (expl[expl.source == "rules"].groupby("transaction_id").factor
               .apply(lambda s: " | ".join(s.head(3))).rename("top_factors"))
queue_cols = ["transaction_id", "timestamp", "period", "customer_id", "card_id", "transaction_type", "channel",
              "entry_mode", "merchant_category", "amount_tnd", "city", "country", "status", "score_model",
              "score_rules", "rank_in_period", "risk_level", "legacy_alert", "is_fraud", "fraud_scenario"]
queue = df.loc[expl_mask, queue_cols].merge(top_factors, on="transaction_id", how="left")
queue["demo_case"] = queue.transaction_id.map({t: k for k, ids in DEMO.items() for t in ids})
save(queue, "gold_alert_queue", null_labels=True)
save(expl, "gold_alert_explanations")
save(pd.DataFrame([{"metric": k, "value": float(val)} for k, val in M.items()]), "gold_model_metrics")
save(per_type.reset_index(), "gold_model_recall_by_type")

submission = df.loc[TE, ["transaction_id", "score_model", "is_alert"]].rename(columns={"score_model": "fraud_score"})
(spark.createDataFrame(submission).coalesce(1).write.mode("overwrite").option("header", True)
      .csv("Files/submission/submission_csv"))
print("written:", len(queue), "queue rows |", len(expl), "explanation rows |", len(submission), "submission rows")

## 7. Checks (paste this output back)

In [ ]:
print("=== CHECK 02 ===")
for t in ["gold_scores", "gold_alert_queue", "gold_alert_explanations", "gold_model_metrics", "gold_model_recall_by_type"]:
    print(f"{t:28s}", spark.table(t).count())
q = spark.table("gold_alert_queue")
q.groupBy("period", "risk_level").count().orderBy("period", "risk_level").show()
print("test rows with a label in queue (must be 0):", q.filter((F.col("period") == "test") & F.col("is_fraud").isNotNull()).count())
q.filter(F.col("demo_case").isNotNull()).select("demo_case", "transaction_id", "amount_tnd", "channel", "country",
                                                "score_model", "score_rules", "top_factors").show(truncate=False)

In [ ]:
mssparkutils.session.stop()   # free the capacity